In [1]:
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
import pickle

from hiv_simulator import HIVSimulator

# Load Dataset

In [2]:
all_episodes = {} 
all_episodes[0.5] = pickle.load(open('batch_trajectories_epsilon=05.p', 'rb'))
all_episodes[0.2] = pickle.load(open('batch_trajectories_epsilon=02.p', 'rb'))

In [3]:
print("Number of trajectories/episodes", len(all_episodes[0.2]))
print("Each episode comprises of {} lists. All lists are the same length. These represent (S, A, R, S', p) tuples".format(len(all_episodes[0.2][0])) )

Number of trajectories/episodes 250
Each episode comprises of 5 lists. All lists are the same length. These represent (S, A, R, S', p) tuples


In [4]:
# Examine the first episode
first_episode = all_episodes[0.2][0]

states = first_episode[0]
actions = first_episode[1]
rewards = first_episode[2]
next_states = first_episode[3]
propensities = first_episode[4] # probability of the selected action under the behavior policy

In [5]:
print( np.array(states[0:5]) )

[[5.21371162 0.69897    4.07718615 1.66275783 4.80562997 1.38021124]
 [5.30317939 1.74918268 2.93439581 1.43190928 3.53918668 1.4207034 ]
 [5.37917686 2.15822386 2.09293134 1.19502454 2.88451327 1.54718007]
 [5.43770298 2.05278582 3.06321797 1.87146423 3.79844829 1.65002342]
 [5.48880868 2.1022221  2.44742931 1.49258374 3.22973987 1.73588493]]


In [6]:
print( np.array(actions[0:5]) )

[3 1 0 1 3]


In [7]:
print( np.array(rewards[0:5]) )

[1.60192273 2.53750502 4.4042061  4.44661169 5.33127257]


In [8]:
print( np.array(next_states[0:5]) )

[[5.30317939 1.74918268 2.93439581 1.43190928 3.53918668 1.4207034 ]
 [5.37917686 2.15822386 2.09293134 1.19502454 2.88451327 1.54718007]
 [5.43770298 2.05278582 3.06321797 1.87146423 3.79844829 1.65002342]
 [5.48880868 2.1022221  2.44742931 1.49258374 3.22973987 1.73588493]
 [5.53366048 2.3834204  1.67247247 0.91825541 2.32646703 1.80149929]]


In [9]:
print( np.array(propensities[0:5]) )

[0.85 0.05 0.85 0.85 0.85]


In [10]:
# Define the horizon H and number of actions (uniform policy)
H = 5  # Horizon (limit to the first 5 steps)
num_actions = 4  # Uniform policy selects each action with equal probability

# (1a)

In [11]:
# Select the dataset for epsilon=0.5 (behavior policy)
episodes = all_episodes[0.5]  # Using the dataset where the behavior policy epsilon=0.5

# Initialize a list to store the weighted returns
returns = []

# Iterate over all episodes
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode

    # Compute total reward for the trajectory up to horizon H
    total_reward = sum(rewards[:H])

    # Compute the importance sampling weight for the trajectory
    # We compute the ratio of probabilities between the uniform policy (πu) and the behavior policy (πb)
    # Since the uniform policy selects each action with equal probability (1/num_actions), 
    # the importance sampling weight for each action is 1 / (num_actions * propensity).
    
    trajectory_weight = np.prod([1 / (num_actions * propensity) for propensity in propensities[:H]])

    # Apply the IS weight to the total reward (weighted return for the trajectory)
    weighted_reward = trajectory_weight * total_reward

    # Store the weighted return
    returns.append(weighted_reward)

# Compute the IS estimate of expected return θ*
theta_hat = np.mean(returns)

# Compute the standard error for the estimate
std_error = np.std(returns) / np.sqrt(len(episodes))

# Construct the 95% confidence interval for the estimate
confidence_interval = (theta_hat - 1.96 * std_error, theta_hat + 1.96 * std_error)

# Output the results
print("Importance Sampling Estimate of Expected Return:", theta_hat)
print("95% Confidence Interval:", confidence_interval)

Importance Sampling Estimate of Expected Return: 16.022712847503588
95% Confidence Interval: (np.float64(10.240930904386312), np.float64(21.804494790620865))


# (1b)

In [12]:
# Select the dataset for epsilon=0.2 (behavior policy)
episodes = all_episodes[0.2]  # Using the dataset where the behavior policy epsilon=0.2

# Initialize a list to store the weighted returns
returns = []

# Iterate over all episodes
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode

    # Compute total reward for the trajectory up to horizon H
    total_reward = sum(rewards[:H])

    # Compute the importance sampling weight for the trajectory
    # We compute the ratio of probabilities between the uniform policy (πu) and the behavior policy (πb)
    # Since the uniform policy selects each action with equal probability (1/num_actions), 
    # the importance sampling weight for each action is 1 / (num_actions * propensity).
    
    trajectory_weight = np.prod([1 / (num_actions * propensity) for propensity in propensities[:H]])

    # Apply the IS weight to the total reward (weighted return for the trajectory)
    weighted_reward = trajectory_weight * total_reward

    # Store the weighted return
    returns.append(weighted_reward)

# Compute the IS estimate of expected return θ*
theta_hat = np.mean(returns)

# Compute the standard error for the estimate
std_error = np.std(returns) / np.sqrt(len(episodes))

# Construct the 95% confidence interval for the estimate
confidence_interval = (theta_hat - 1.96 * std_error, theta_hat + 1.96 * std_error)

# Output the results
print("Importance Sampling Estimate of Expected Return:", theta_hat)
print("95% Confidence Interval:", confidence_interval)

Importance Sampling Estimate of Expected Return: 17.487166243967142
95% Confidence Interval: (np.float64(-8.804967590971582), np.float64(43.77930007890586))


# (1c)

In [13]:
# Select the dataset for epsilon=0.5 (behavior policy)
episodes_epsilon_05 = all_episodes[0.5]  

# Calculate returns with importance sampling for epsilon=0.5
returns_epsilon_05 = []
weights_epsilon_05 = []

for episode in episodes_epsilon_05:
    states, actions, rewards, next_states, propensities = episode

    # Compute total reward for the trajectory up to horizon H
    total_reward = sum(rewards[:H])

    # Compute the importance sampling weight (product of propensities)
    trajectory_weight = np.prod([1 / (num_actions * propensity) for propensity in propensities[:H]])

    # Store the weighted return and the weight for ESS computation
    returns_epsilon_05.append(trajectory_weight * total_reward)
    weights_epsilon_05.append(trajectory_weight)

# Compute ESS for epsilon=0.5
ESS_epsilon_05 = (np.sum(weights_epsilon_05)**2) / np.sum(np.square(weights_epsilon_05))

# Select the dataset for epsilon=0.2 (behavior policy)
episodes_epsilon_02 = all_episodes[0.2]

# Calculate returns with importance sampling for epsilon=0.2
returns_epsilon_02 = []
weights_epsilon_02 = []

for episode in episodes_epsilon_02:
    states, actions, rewards, next_states, propensities = episode

    # Compute total reward for the trajectory up to horizon H
    total_reward = sum(rewards[:H])

    # Compute the importance sampling weight (product of propensities)
    trajectory_weight = np.prod([1 / (num_actions * propensity) for propensity in propensities[:H]])

    # Store the weighted return and the weight for ESS computation
    returns_epsilon_02.append(trajectory_weight * total_reward)
    weights_epsilon_02.append(trajectory_weight)

# Compute ESS for epsilon=0.2
ESS_epsilon_02 = (np.sum(weights_epsilon_02)**2) / np.sum(np.square(weights_epsilon_02))

# Output the ESS values for both epsilon=0.5 and epsilon=0.2
print("Effective Sample Size for epsilon=0.5:", ESS_epsilon_05)
print("Effective Sample Size for epsilon=0.2:", ESS_epsilon_02)

Effective Sample Size for epsilon=0.5: 23.615677214314246
Effective Sample Size for epsilon=0.2: 1.7810722282019509


# (1d)

In [14]:
# Select the dataset for epsilon=0.5 (behavior policy)
episodes_epsilon_05 = all_episodes[0.5]

# Calculate returns with weighted importance sampling for epsilon=0.5
weighted_returns = []
weights = []

for episode in episodes_epsilon_05:
    states, actions, rewards, next_states, propensities = episode

    # Compute total reward for the trajectory up to horizon H
    total_reward = sum(rewards[:H])

    # Compute the importance sampling weight (product of the inverse of the propensities for each action)
    trajectory_weight = np.prod([1 / (num_actions * propensity) for propensity in propensities[:H]])

    # Store the weighted return and the weight for normalization
    weighted_returns.append(trajectory_weight * total_reward)
    weights.append(trajectory_weight)

# Compute the weighted importance sampling estimate
numerator = np.sum(weighted_returns)
denominator = np.sum(weights)

# Final weighted importance sampling estimate
theta_hat_WIS = numerator / denominator

# Compute standard error for WIS
std_error_WIS = np.std(weighted_returns) / np.sqrt(len(episodes_epsilon_05))

# Construct 95% confidence interval for WIS
confidence_interval_WIS = (theta_hat_WIS - 1.96 * std_error_WIS, theta_hat_WIS + 1.96 * std_error_WIS)

# Output the results for WIS
print("Weighted Importance Sampling Estimate of Expected Return:", theta_hat_WIS)
print("95% Confidence Interval (WIS):", confidence_interval_WIS)

Weighted Importance Sampling Estimate of Expected Return: 13.198579960810633
95% Confidence Interval (WIS): (np.float64(7.416798017693358), np.float64(18.980361903927907))
